In [0]:
%python
# Databricks notebook source
# PURPOSE:
# Minimal "agentic" reasoning outputs:
# - capability gaps
# - medical deserts
# - anomaly candidates
# - citation-backed recommendation payload

from pyspark.sql import functions as F

CATALOG = "vf_health"
SCHEMA = "ghana"

SILVER = f"{CATALOG}.{SCHEMA}.silver_facilities_clean"
DESERTS = f"{CATALOG}.{SCHEMA}.gold_medical_deserts"
RISKS = f"{CATALOG}.{SCHEMA}.gold_risk_signals"
CITATIONS = f"{CATALOG}.{SCHEMA}.gold_citations"

OUT = f"{CATALOG}.{SCHEMA}.gold_agent_recommendations"

silver = spark.table(SILVER)
deserts = spark.table(DESERTS)
risks = spark.table(RISKS)
cites = spark.table(CITATIONS)

# 1) Priority regions by desert_score
priority_regions = deserts.orderBy(F.col("desert_score").desc()).limit(10)

# 2) Risk summary by region
risk_with_region = (
    risks.join(
        silver.select("row_id", F.coalesce("address_stateOrRegion", F.lit("UNKNOWN")).alias("region")),
        on="row_id",
        how="left"
    )
    .groupBy("region")
    .agg(
        F.sum(F.when(F.col("signal_flag"), 1).otherwise(0)).alias("risk_flags"),
        F.avg("signal_score").alias("avg_risk_score")
    )
)

# 3) Build recommendation text
recommendations = (
    priority_regions.join(risk_with_region, on="region", how="left")
    .withColumn(
        "recommendation",
        F.concat(
            F.lit("Prioritize region "),
            F.col("region"),
            F.lit(" / district "),
            F.col("district"),
            F.lit(" for emergency + maternal capability investments. "),
            F.lit("Desert score="),
            F.round("desert_score", 3).cast("string"),
            F.lit(", risk flags="),
            F.coalesce(F.col("risk_flags").cast("string"), F.lit("0"))
        )
    )
    .withColumn("created_at", F.current_timestamp())
)

recommendations.write.mode("overwrite").format("delta").saveAsTable(OUT)

print("Agent recommendations rows:", spark.table(OUT).count())
display(spark.table(OUT).limit(20))